# IDX COLAB GPU TRAINING CENTER

**Peran:** training + research eksternal (bukan production runtime)

Architecture freeze: production pointer **tidak** diubah dari Colab secara default.


## 1. Environment Check


In [ ]:
import sys, os, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())


## 2. Connect Google Drive (optional)


In [ ]:
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/IDX')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('DRIVE = PASS', DRIVE)
except Exception as e:
    print('DRIVE = BLOCKED', type(e).__name__)


## 3. Clone / Update Repository


In [ ]:
import os, subprocess
from pathlib import Path
REPO = 'https://github.com/whatman42/idx.git'
ROOT = Path('/content/idx')
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only'])
os.chdir(ROOT)
print('cwd', os.getcwd())
sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', sha)


## 4. Install Dependencies


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('deps installed')


## 5. GPU Check


In [ ]:
import sys
sys.path.insert(0, '/content/idx')
from colab.environment_check import check_environment
env = check_environment()
print('GPU_AVAILABLE =', env.gpu_available)
print('GPU:', env.gpu_name, 'VRAM GB:', env.gpu_vram_gb)
print('Status:', env.status, env.notes)


## 6-16. Training Pipeline


In [ ]:
from colab.colab_config import ColabConfig, DatasetType
from colab.colab_train import run_colab_training
cfg = ColabConfig(out_dir='artifacts/colab_candidates', dataset_type=DatasetType.SYNTHETIC_DATA, n_bars=120, promote=False)
report = run_colab_training(cfg)
print(report.summary_text())
print('MARKET PERFORMANCE =', report.market_performance)
print('GPU LIVE =', report.gpu_live)
print('artifact:', report.artifact_dir)


## 17. SHA256 verify


In [ ]:
from pathlib import Path
from colab.artifact_export import verify_bundle
if report.artifact_dir:
    ok, reason = verify_bundle(Path(report.artifact_dir))
    print('SHA256:', 'PASS' if ok else 'FAIL', reason)
else:
    print('SHA256: SKIP')


## 18. Final Status


In [ ]:
print(report.summary_text())
assert report.production_unchanged
print('PRODUCTION POINTER: UNCHANGED')
